# Preprocesamiento común de egresos hospitalarios 2025

Exploración, revisión de calidad y preparación de **una misma base para todo el equipo**.
Se conservan las variables necesarias para estudiar condición al egreso, duración, perfiles y patrones.
Cada fila representa un egreso; no necesariamente una persona distinta.

El notebook no selecciona predictores ni variables objetivo, no particiona, no codifica para modelos
y no aplica imputaciones estadísticas. Las preparaciones específicas se harán posteriormente.

Ejecutar las celdas en orden desde la carpeta del proyecto, con las dependencias de `requirements.txt`.
La fuente está en `datos/originales/EGRESOS_2025.zip`. La única base resultante se guarda como
`datos/procesados/egresos_2025_base_comun.parquet`.

In [ ]:
%pip install -r requirements.txt

## 1. Carga del conjunto original y del diccionario

El CSV usa punto y coma y UTF-8. Se conservan los códigos como texto para no perder ceros iniciales.
La lectura mantiene vacíos, `*` y el texto `NULL` tal como vienen. La huella SHA-256 identifica el ZIP.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import hashlib
import platform
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RAIZ = Path.cwd()
ARCHIVO_ZIP = RAIZ / "datos" / "originales" / "EGRESOS_2025.zip"
PROCESADOS = RAIZ / "datos" / "procesados"
TABLAS = RAIZ / "resultados" / "tablas"
FIGURAS = RAIZ / "resultados" / "figuras"
for carpeta in [PROCESADOS, TABLAS, FIGURAS]:
    carpeta.mkdir(parents=True, exist_ok=True)

UMBRAL_REVISION_DIAS = 365
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 85)
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})

paquetes = ["pandas", "numpy", "matplotlib", "openpyxl", "pyarrow"]
versiones = {paquete: version(paquete) for paquete in paquetes}
display(pd.Series({"Python": platform.python_version(), **versiones}, name="versión").to_frame())

In [ ]:
with ZipFile(ARCHIVO_ZIP) as archivo:
    contenido = archivo.read("EGR_DATOS_ABIERTO_2025.csv")

print("Hualaihué:", b"Hualaihu\xc3\xa9" in contenido)
print("Hualaihu�:", b"Hualaihu\xef\xbf\xbd" in contenido)

Se leen ambos archivos desde el ZIP, sin modificar ni extraer la fuente original.
El diccionario denomina `DIAS_ESTAD` al campo que aparece como `DIAS_ESTADA` en el CSV.

In [ ]:
with ZipFile(ARCHIVO_ZIP) as archivo:
    contenido_zip = pd.DataFrame({"archivo": archivo.namelist(),
                                 "bytes_sin_comprimir": [a.file_size for a in archivo.infolist()]})
    with archivo.open("EGR_DATOS_ABIERTO_2025.csv") as csv:
        originales = pd.read_csv(csv, sep=";", encoding="utf-8", dtype="string", keep_default_na=False)
    libro = BytesIO(archivo.read("Diccionario BD egresos hospitalario.xlsx"))
    diccionario = pd.read_excel(libro, sheet_name="Hoja1", dtype="string")
    cie10 = pd.read_excel(libro, sheet_name="codigo CIE-10", header=8, dtype="string")

sha256 = hashlib.sha256(ARCHIVO_ZIP.read_bytes()).hexdigest()
print(f"Registros: {len(originales):,} | Variables originales: {originales.shape[1]}")
print("SHA-256 del ZIP:", sha256)
display(contenido_zip)
display(originales.head(3))

In [ ]:
diccionario.columns = ["variable", "descripcion_diccionario"]
diccionario["variable"] = diccionario["variable"].str.strip().replace({"DIAS_ESTAD": "DIAS_ESTADA"})
descripcion_variables = pd.DataFrame({"variable": originales.columns})
descripcion_variables = descripcion_variables.merge(diccionario, on="variable", how="left")
descripcion_variables["descripcion_diccionario"] = descripcion_variables["descripcion_diccionario"].fillna(
    "No definida en el diccionario entregado; significado pendiente de confirmar")
descripcion_variables["tipo_lectura"] = originales.dtypes.astype(str).values
descripcion_variables["valores_distintos"] = originales.nunique().values
descripcion_variables["rol"] = "Categórica o código; no representa una cantidad"
descripcion_variables.loc[descripcion_variables["variable"].eq("DIAS_ESTADA"), "rol"] = "Cantidad de días de estadía"
descripcion_variables.loc[descripcion_variables["variable"].eq("ANO_EGRESO"), "rol"] = "Año de referencia"
descripcion_variables.loc[descripcion_variables["variable"].eq("ERROR"), "rol"] = "Indicador de significado no documentado"
display(descripcion_variables)

`DIAG2` corresponde a causa externa, no a una segunda enfermedad. Sus vacíos pueden ser estructurales.
La edad es agrupada y la pertenencia al SNSS no equivale a identificar hospitales públicos o clínicas privadas.

## 2. Exploración inicial

Se muestran dimensiones, atributos, tipos de lectura, frecuencias y estadísticas descriptivas.
El detalle de 1–60 días es un acercamiento gráfico; la distribución completa incluye las estadías extremas.

In [ ]:
dias_originales = pd.to_numeric(originales["DIAS_ESTADA"], errors="coerce")
resumen_estadia = dias_originales.describe(percentiles=[.25, .50, .75, .90, .95, .99]).to_frame("dias")
frecuencia_condicion = originales["CONDICION_EGRESO"].value_counts().rename_axis("codigo").to_frame("registros")
frecuencia_condicion["condicion"] = frecuencia_condicion.index.map({"1": "Vivo", "2": "Fallecido"})
frecuencia_condicion["porcentaje"] = 100 * frecuencia_condicion["registros"] / len(originales)
display(resumen_estadia.round(3))
display(frecuencia_condicion.round(3))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].hist(dias_originales[dias_originales.between(1, 60)], bins=np.arange(.5, 61.5), color="#30759b")
ejes[0].set(title="Detalle de estadías de 1 a 60 días", xlabel="Días de estadía", ylabel="Número de egresos")
ejes[1].hist(dias_originales.dropna(), bins=np.geomspace(1, dias_originales.max() + 1, 55), color="#30759b")
ejes[1].set_xscale("log")
ejes[1].set_yscale("log")
ejes[1].set(title="Distribución completa, ejes logarítmicos", xlabel="Días de estadía", ylabel="Número de egresos")
fig.tight_layout()
fig.savefig(FIGURAS / "01_distribucion_estadia.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Revisión de calidad

Se distinguen vacíos, `*`, texto literal `NULL`, caracteres dañados, repeticiones y valores extremos.
En este CSV, `NULL` en región se asocia con la glosa `Extranjero`; se conserva como categoría de origen.
`*` y `ERROR=1` no tienen significado confirmado en el diccionario y no son motivos de exclusión.

In [ ]:
calidad_inicial = pd.DataFrame({
    "vacios": originales.eq("").sum(),
    "asteriscos": originales.eq("*").sum(),
    "texto_NULL": originales.eq("NULL").sum(),
    "caracter_danado": originales.apply(lambda columna: columna.str.contains("\ufffd", regex=False).sum()),
    "valores_distintos": originales.nunique(),
}).rename_axis("variable")
calidad_inicial["porcentaje_vacio"] = 100 * calidad_inicial["vacios"] / len(originales)
calidad_inicial["porcentaje_asterisco"] = 100 * calidad_inicial["asteriscos"] / len(originales)
display(calidad_inicial.round(2))
display(originales.loc[originales["REGION_RESIDENCIA"].eq("NULL"),
                       ["REGION_RESIDENCIA", "GLOSA_REGION_RESIDENCIA"]].value_counts().to_frame("registros"))

In [ ]:
variables_explorar = ["PERTENENCIA_ESTABLECIMIENTO_SALU", "SEXO", "GRUPO_EDAD", "ETNIA",
                     "GLOSA_PAIS_ORIGEN", "REGION_RESIDENCIA", "PREVISION", "ANO_EGRESO", "ERROR"]
frecuencias = []
for variable in variables_explorar:
    tabla = originales[variable].value_counts(dropna=False).rename_axis("valor").reset_index(name="registros")
    tabla.insert(0, "variable", variable)
    tabla["porcentaje"] = 100 * tabla["registros"] / len(originales)
    frecuencias.append(tabla)
frecuencias_categoricas = pd.concat(frecuencias, ignore_index=True)
display(frecuencias_categoricas.round(3))

fig, ax = plt.subplots(figsize=(10, 4))
conteos = calidad_inicial.loc[calidad_inicial["asteriscos"].gt(0), "asteriscos"]
conteos.plot.barh(ax=ax, color="#b67932")
ax.set(title="Marcador * por variable", xlabel="Número de egresos", ylabel="")
fig.tight_layout()
fig.savefig(FIGURAS / "02_asteriscos_por_variable.png", dpi=160, bbox_inches="tight")
plt.show()

Las filas con los 17 campos iguales se cuentan y se conservan: pueden corresponder a egresos distintos.
Sin identificador de paciente no se puede decidir que una repetición sea un registro duplicado por error.

In [ ]:
repetida_adicional = originales.duplicated()
en_grupo_repetido = originales.duplicated(keep=False)
resumen_repeticiones = pd.Series({
    "filas_totales": len(originales),
    "combinaciones_distintas_de_los_17_campos": len(originales) - int(repetida_adicional.sum()),
    "repeticiones_adicionales": int(repetida_adicional.sum()),
    "filas_que_pertenecen_a_grupos_repetidos": int(en_grupo_repetido.sum()),
}, name="registros").to_frame()
display(resumen_repeticiones)

## 4. Transformaciones comunes

Se retiran espacios externos, se representan vacíos y `*` como ausentes y se conservan marcas de origen.
Esto expresa la falta de información, sin atribuirle una causa ni imputar valores. `NULL` permanece intacto.
Se corrigen dos etiquetas de edad identificables. Las glosas geográficas de origen se conservan;
se añaden versiones corregidas por código mediante la referencia oficial documentada abajo.
Las conversiones numéricas se contabilizan aparte de los cambios de contenido.

In [ ]:
datos = originales.copy()
datos.insert(0, "ID_REGISTRO", np.arange(1, len(datos) + 1, dtype=np.int64))
datos["TIENE_ASTERISCO"] = originales.eq("*").any(axis=1)
datos["TIENE_TEXTO_DANADO"] = originales.apply(
    lambda columna: columna.str.contains("\ufffd", regex=False)).any(axis=1)
datos["EN_GRUPO_REPETIDO"] = en_grupo_repetido.to_numpy()
datos["DIAG2_VACIO_ORIGEN"] = originales["DIAG2"].eq("")

sin_espacios = originales.apply(lambda columna: columna.str.strip())
cambio_espacios = originales.ne(sin_espacios)
vacios = sin_espacios.eq("")
asteriscos = sin_espacios.eq("*")
datos[originales.columns] = sin_espacios.replace({"": pd.NA, "*": pd.NA})

etiquetas_edad = {"menor de un a\ufffdo": "Menor de 1 año", "90 y m\ufffds": "90 y más"}
cambio_edad = datos["GRUPO_EDAD"].isin(etiquetas_edad)
datos["GRUPO_EDAD"] = datos["GRUPO_EDAD"].replace(etiquetas_edad)
datos["TEXTO_DANADO_PENDIENTE"] = datos[["GLOSA_COMUNA_RESIDENCIA", "GLOSA_REGION_RESIDENCIA"]].apply(
    lambda columna: columna.str.contains("\ufffd", regex=False, na=False)).any(axis=1)

columnas_numericas = ["DIAS_ESTADA", "ANO_EGRESO", "CONDICION_EGRESO", "ERROR"]
conversiones_tipo = 0
valores_no_convertibles = {}
for columna in columnas_numericas:
    valores = pd.to_numeric(datos[columna], errors="coerce")
    valores_no_convertibles[columna] = int((datos[columna].notna() & valores.isna()).sum())
    conversiones_tipo += int(valores.notna().sum())
    datos[columna] = valores.astype("Int64")

codigos_antes = datos[["DIAG1", "DIAG2"]].copy()
for columna in ["DIAG1", "DIAG2"]:
    datos[columna] = datos[columna].str.upper().str.replace(".", "", regex=False)
cambio_codigos = codigos_antes.ne(datos[["DIAG1", "DIAG2"]]).fillna(False)

transformaciones = pd.DataFrame([
    ["Retirar espacios externos", "Evitar diferencias por formato", int(cambio_espacios.sum().sum()), int(cambio_espacios.any(axis=1).sum())],
    ["Vacíos a ausentes", "Representar ausencia sin inventar valores", int(vacios.sum().sum()), int(vacios.any(axis=1).sum())],
    ["Asteriscos a ausentes", "Mantener la falta de información y su marca de origen", int(asteriscos.sum().sum()), int(asteriscos.any(axis=1).sum())],
    ["Corregir etiquetas de edad", "Recuperar dos intervalos identificables", int(cambio_edad.sum()), int(cambio_edad.sum())],
    ["Uniformar códigos CIE-10", "Usar mayúsculas y códigos sin puntos para el cruce", int(cambio_codigos.sum().sum()), int(cambio_codigos.any(axis=1).sum())],
    ["Convertir tipos numéricos", "Representar cantidades y códigos según el diccionario", conversiones_tipo, int(datos[columnas_numericas].notna().any(axis=1).sum())],
], columns=["transformacion", "motivo", "celdas_afectadas", "registros_afectados"])
display(transformaciones)
display(pd.Series(valores_no_convertibles, name="valores_no_convertibles").to_frame())

### Corrección de glosas geográficas mediante códigos oficiales

El CSV ya contiene `�` (U+FFFD): UTF-8 permite leerlo, pero no recuperar la letra perdida.
Se conservan las glosas existentes y se crean versiones con sufijo `_CORREGIDA` mediante el código
de comuna o región. El estado distingue reparación de caracteres, normalización del nombre,
coincidencia, categoría especial y ausencia de correspondencia.

**Fuente:** [capa DPA publicada por la SMA](https://ideserver.sma.gob.cl/arcgis/rest/services/IDE/Limites/MapServer/2),
con metadatos del Grupo de Trabajo de División Político Administrativa (2023), integrado por
DIFROL, SUBDERE, IGM, INE y CIREN. Campos: `CUT_COM`, `COMUNA`, `CUT_REG`, `REGION`.
Referencia de códigos: [CUT de SUBDERE, actualización 2018](https://www.subdere.gov.cl/documentacion/c%C3%B3digos-%C3%BAnicos-territoriales-actualizados-al-06-de-septiembre-2018).

Consulta: **2026-11-08**. Se conserva una copia CSV de 345 comunas y 16 regiones para ejecución sin conexión.
La capa no incluye Antártica (`12202`); no hay filas con ese código en esta fuente de egresos.
[Consulta de origen](https://ideserver.sma.gob.cl/arcgis/rest/services/IDE/Limites/MapServer/2/query?where=1%3D1&outFields=CUT_COM%2CCOMUNA%2CCUT_REG%2CREGION&returnGeometry=false&f=json). La huella de la copia utilizada se muestra al leerla.

`99999`/`99` (Ignorada), `88888`/`NULL` (Extranjero) y los ausentes se conservan, sin asignarles
territorios nacionales. Un código no encontrado mantiene su glosa y se informa para revisión.
Cambios como `De Los Lagos → Los Lagos` se cuentan como normalización, separados de la reparación de `�`.

In [ ]:
archivo_geografico = RAIZ / "datos" / "originales" / "catalogo_geografico_dpa_2023.csv"
catalogo_geografico = pd.read_csv(archivo_geografico, dtype="string", encoding="utf-8-sig")
mapa_comunas = catalogo_geografico.set_index("CUT_COM")["COMUNA"]
mapa_regiones = catalogo_geografico[["CUT_REG", "REGION"]].drop_duplicates().set_index("CUT_REG")["REGION"]
assert mapa_comunas.index.is_unique and mapa_regiones.index.is_unique, "Cada código debe tener un solo nombre."
print("SHA-256 del catálogo geográfico:", hashlib.sha256(archivo_geografico.read_bytes()).hexdigest())

cambios_geograficos = pd.DataFrame(index=datos.index)
filas_transformacion_geografica = []
campos_geograficos = [
    ("COMUNA", "COMUNA_RESIDENCIA", "GLOSA_COMUNA_RESIDENCIA", mapa_comunas, ["99999", "88888"]),
    ("REGION", "REGION_RESIDENCIA", "GLOSA_REGION_RESIDENCIA", mapa_regiones, ["99", "NULL"]),
]
for nivel, codigo, glosa, correspondencia, especiales in campos_geograficos:
    nombre_oficial = datos[codigo].map(correspondencia)
    datos[glosa + "_CORREGIDA"] = nombre_oficial.fillna(datos[glosa])
    danado = datos[glosa].str.contains("\ufffd", regex=False, na=False)
    cambiado = nombre_oficial.notna() & datos[glosa].ne(nombre_oficial).fillna(True)
    estado = pd.Series("SIN_CORRESPONDENCIA", index=datos.index, dtype="string")
    estado.loc[datos[codigo].isna()] = "SIN_DATO"
    estado.loc[datos[codigo].isin(especiales)] = "CATEGORIA_ESPECIAL"
    estado.loc[nombre_oficial.notna()] = "COINCIDE"
    estado.loc[cambiado & ~danado] = "NORMALIZADA"
    estado.loc[cambiado & danado] = "CORREGIDA"
    datos["ESTADO_CORRECCION_" + nivel] = estado
    cambios_geograficos[nivel] = cambiado
    cantidad = int(cambiado.sum())
    filas_transformacion_geografica.append([
        "Crear glosa corregida de " + nivel.lower(),
        "Cruce por código con DPA 2023; se conserva la glosa de origen", cantidad, cantidad])

glosas_corregidas = ["GLOSA_COMUNA_RESIDENCIA_CORREGIDA", "GLOSA_REGION_RESIDENCIA_CORREGIDA"]
datos["TEXTO_DANADO_PENDIENTE"] = datos[glosas_corregidas].apply(
    lambda columna: columna.str.contains("\ufffd", regex=False, na=False)).any(axis=1)
resumen_geografico = pd.DataFrame({nivel: datos["ESTADO_CORRECCION_" + nivel].value_counts()
                                  for nivel in ["COMUNA", "REGION"]})
resumen_geografico = resumen_geografico.reindex(
    ["CORREGIDA", "NORMALIZADA", "COINCIDE", "CATEGORIA_ESPECIAL", "SIN_DATO", "SIN_CORRESPONDENCIA"]
).fillna(0).astype(int).rename_axis("estado")
transformaciones = pd.concat([
    transformaciones, pd.DataFrame(filas_transformacion_geografica, columns=transformaciones.columns)
], ignore_index=True)
display(resumen_geografico)
for nivel, codigo, glosa, correspondencia, especiales in campos_geograficos:
    pendientes = datos.loc[datos["ESTADO_CORRECCION_" + nivel].eq("SIN_CORRESPONDENCIA"), [codigo, glosa]]
    print("Sin correspondencia de " + nivel.lower() + ":", len(pendientes))
    display(pendientes.value_counts(dropna=False).to_frame("registros"))
display(datos.loc[datos["COMUNA_RESIDENCIA"].eq("10403"),
                  ["COMUNA_RESIDENCIA", "GLOSA_COMUNA_RESIDENCIA", "GLOSA_COMUNA_RESIDENCIA_CORREGIDA"]]
        .value_counts().to_frame("registros"))
display(
    datos.loc[
        datos["ESTADO_CORRECCION_COMUNA"] == "CORREGIDA",
        ["GLOSA_COMUNA_RESIDENCIA", "GLOSA_COMUNA_RESIDENCIA_CORREGIDA"]
    ].drop_duplicates().sort_values("GLOSA_COMUNA_RESIDENCIA_CORREGIDA")
)

### Incorporación de información CIE-10

Se normalizan los códigos del diccionario y se incorpora la categoría, descripción y capítulo del diagnóstico
principal. Es un enriquecimiento descriptivo; no selecciona atributos para un algoritmo. Los códigos completos
se conservan. Se distingue coincidencia exacta, coincidencia solo de categoría y ausencia de correspondencia.

In [ ]:
cie10.columns = ["codigo", "diagnostico", "capitulo", "glosa_capitulo", "grupo", "glosa_grupo",
                 "categoria", "glosa_categoria"]
sin_cruce_antes = int((~originales["DIAG1"].isin(cie10["codigo"])).sum())
cie10 = cie10.dropna(subset=["codigo"]).copy()
for columna in cie10.columns:
    cie10[columna] = cie10[columna].str.strip()
for columna in ["codigo", "categoria", "grupo", "capitulo"]:
    cie10[columna] = cie10[columna].str.upper().str.replace(".", "", regex=False)
cie_por_codigo = cie10.set_index("codigo")
categorias_cie = cie10[["categoria", "glosa_categoria", "capitulo", "glosa_capitulo"]].drop_duplicates()
categorias_cie = categorias_cie.set_index("categoria")

datos["DIAG1_CATEGORIA"] = datos["DIAG1"].str[:3]
datos["DIAG1_DESCRIPCION"] = datos["DIAG1"].map(cie_por_codigo["diagnostico"])
datos["DIAG1_GLOSA_CATEGORIA"] = datos["DIAG1_CATEGORIA"].map(categorias_cie["glosa_categoria"])
datos["DIAG1_CAPITULO"] = datos["DIAG1_CATEGORIA"].map(categorias_cie["capitulo"])
datos["DIAG1_GLOSA_CAPITULO"] = datos["DIAG1_CATEGORIA"].map(categorias_cie["glosa_capitulo"])
datos["DIAG1_COINCIDENCIA"] = "SIN_COINCIDENCIA"
datos.loc[datos["DIAG1_CATEGORIA"].isin(categorias_cie.index), "DIAG1_COINCIDENCIA"] = "SOLO_CATEGORIA"
datos.loc[datos["DIAG1"].isin(cie_por_codigo.index), "DIAG1_COINCIDENCIA"] = "SUBCATEGORIA_EXACTA"
datos.loc[datos["DIAG1"].isna(), "DIAG1_COINCIDENCIA"] = "SIN_DATO"
datos["DIAG2_COINCIDENCIA"] = "SIN_COINCIDENCIA"
datos.loc[datos["DIAG2"].isin(cie_por_codigo.index), "DIAG2_COINCIDENCIA"] = "SUBCATEGORIA_EXACTA"
datos.loc[datos["DIAG2"].isna(), "DIAG2_COINCIDENCIA"] = "SIN_DATO"

diagnosticos_sin_coincidencia = (
    datos.loc[datos["DIAG1"].notna() & ~datos["DIAG1"].isin(cie_por_codigo.index),
              ["DIAG1", "DIAG1_CATEGORIA", "DIAG1_COINCIDENCIA"]]
    .value_counts().reset_index(name="registros")
)
resumen_cruce_cie = datos["DIAG1_COINCIDENCIA"].value_counts().to_frame("registros")
comparacion_cruce_cie = pd.Series({
    "sin_coincidencia_exacta_antes_de_normalizar": sin_cruce_antes,
    "sin_coincidencia_exacta_despues_de_normalizar": int(diagnosticos_sin_coincidencia["registros"].sum()),
}, name="registros").to_frame()
display(comparacion_cruce_cie)
display(resumen_cruce_cie)
display(diagnosticos_sin_coincidencia)

sin_cruce_diag2 = datos.loc[datos["DIAG2"].notna() & ~datos["DIAG2"].isin(cie_por_codigo.index), "DIAG2"]
sin_cruce_diag2 = sin_cruce_diag2.value_counts().rename_axis("codigo").reset_index(name="registros")
diagnosticos_pendientes = diagnosticos_sin_coincidencia.rename(columns={"DIAG1": "codigo"})[["codigo", "registros"]].copy()
diagnosticos_pendientes.insert(0, "variable", "DIAG1")
sin_cruce_diag2.insert(0, "variable", "DIAG2")
diagnosticos_pendientes = pd.concat([diagnosticos_pendientes, sin_cruce_diag2], ignore_index=True)
display(datos["DIAG2_COINCIDENCIA"].value_counts().to_frame("registros"))


Los códigos no encontrados se conservan para revisión: no se declaran inválidos. La marca de coincidencia
de `DIAG2` distingue ausencia de causa externa y código sin correspondencia. El diagnóstico principal es
información del egreso, lo que limita su interpretación en los análisis posteriores.

### Verificación de inconsistencias y marcas de revisión

Más de 365 días es un umbral operativo de revisión, no una definición clínica ni una regla de exclusión.
Se cuentan valores no positivos, códigos inesperados y contradicciones entre códigos y glosas; se reportan
sin aplicar correcciones cuyo significado no esté documentado.

In [ ]:
datos["ESTADIA_MAYOR_365"] = datos["DIAS_ESTADA"].gt(UMBRAL_REVISION_DIAS).fillna(False)
resumen_extremos = pd.DataFrame({
    "criterio": ["Más de 60 días", "Más de 365 días", "Más de 1000 días"],
    "registros": [int(datos["DIAS_ESTADA"].gt(60).sum()), int(datos["ESTADIA_MAYOR_365"].sum()),
                  int(datos["DIAS_ESTADA"].gt(1000).sum())],
})
resumen_extremos["porcentaje"] = 100 * resumen_extremos["registros"] / len(datos)
inconsistencias = pd.Series({
    "dias_no_positivos": int(datos["DIAS_ESTADA"].le(0).sum()),
    "dias_no_convertibles": valores_no_convertibles["DIAS_ESTADA"],
    "ano_distinto_de_2025_entre_conocidos": int((datos["ANO_EGRESO"].notna() & datos["ANO_EGRESO"].ne(2025)).sum()),
    "condicion_fuera_de_1_y_2": int((datos["CONDICION_EGRESO"].notna() & ~datos["CONDICION_EGRESO"].isin([1, 2])).sum()),
    "ERROR_fuera_de_0_y_1": int((datos["ERROR"].notna() & ~datos["ERROR"].isin([0, 1])).sum()),
}, name="registros").to_frame()

coherencia_codigos = []
for codigo, glosa in [("COMUNA_RESIDENCIA", "GLOSA_COMUNA_RESIDENCIA"),
                       ("REGION_RESIDENCIA", "GLOSA_REGION_RESIDENCIA"), ("PREVISION", "GLOSA_PREVISION")]:
    pares = datos[[codigo, glosa]].dropna().drop_duplicates()
    glosas_por_codigo = pares.groupby(codigo)[glosa].nunique()
    codigos_ambiguos = glosas_por_codigo[glosas_por_codigo.gt(1)].index
    coherencia_codigos.append({"variable": codigo, "codigos_con_multiples_glosas": len(codigos_ambiguos),
                              "registros_afectados": int(datos[codigo].isin(codigos_ambiguos).sum())})
coherencia_codigos = pd.DataFrame(coherencia_codigos)
display(resumen_extremos.round(4))
display(inconsistencias)
display(coherencia_codigos)
display(datos.nlargest(10, "DIAS_ESTADA")[["ID_REGISTRO", "DIAS_ESTADA", "DIAG1", "GRUPO_EDAD", "ERROR"]])

comparacion_marcadores = datos.groupby("TIENE_ASTERISCO").agg(
    registros=("ID_REGISTRO", "size"), media_dias=("DIAS_ESTADA", "mean"), mediana_dias=("DIAS_ESTADA", "median"))
display(comparacion_marcadores.round(3))

### Descripción de la base preparada

Se conservan los perfiles descriptivos por edad, sexo y categoría diagnóstica. Las medias se acompañan
de medianas y tamaños de grupo porque las estadías extremas influyen en ellas.

In [ ]:
orden_edad = ["Menor de 1 año", "1 a 9", "10 a 19", "20 a 29", "30 a 39", "40 a 49",
              "50 a 59", "60 a 69", "70 a 79", "80 a 89", "90 y más", "SIN_DATO"]
perfil_edad = datos.assign(edad=datos["GRUPO_EDAD"].fillna("SIN_DATO")).groupby("edad").agg(
    registros=("ID_REGISTRO", "size"), media_dias=("DIAS_ESTADA", "mean"), mediana_dias=("DIAS_ESTADA", "median")
).reindex(orden_edad)
perfil_diagnostico = datos.groupby(["DIAG1_CATEGORIA", "DIAG1_GLOSA_CATEGORIA"], dropna=False).agg(
    registros=("ID_REGISTRO", "size"), media_dias=("DIAS_ESTADA", "mean"), mediana_dias=("DIAS_ESTADA", "median")
).sort_values("registros", ascending=False)
perfil_sexo = datos.groupby("SEXO", dropna=False).agg(
    registros=("ID_REGISTRO", "size"), media_dias=("DIAS_ESTADA", "mean"), mediana_dias=("DIAS_ESTADA", "median"))
display(perfil_edad.round(3))
display(perfil_diagnostico.head(15).round(3))
display(perfil_sexo.round(3))

fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
perfil_edad["registros"].plot.barh(ax=ejes[0], color="#30759b")
ejes[0].set(title="Egresos por grupo de edad", xlabel="Número de egresos", ylabel="")
perfil_edad[["media_dias", "mediana_dias"]].plot.barh(ax=ejes[1], color=["#30759b", "#b67932"])
ejes[1].set(title="Estadía por grupo de edad", xlabel="Días", ylabel="")
ejes[1].legend(["Media", "Mediana"])
fig.tight_layout()
fig.savefig(FIGURAS / "03_edad_y_estadia.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Verificación del resultado

Se comprueba la conservación de filas y variables, los ausentes y los cambios de contenido.
Una fila se cuenta como modificada si cambia algún valor o la representación de un faltante en los
17 campos originales. La conversión de tipo con el mismo valor textual y las columnas nuevas se
contabilizan aparte. Las filas afectadas por distintas transformaciones pueden coincidir; no se suman.
En la tabla de calidad, las medidas del original quedan vacías para las columnas añadidas.

Los cambios de las glosas derivadas se cuentan aparte; las glosas originales no se sustituyen.

In [ ]:
comparables = datos[originales.columns].astype("string").fillna("<AUSENTE>")
cambios_contenido = originales.ne(comparables)
registros_modificados = int(cambios_contenido.any(axis=1).sum())
calidad_final = calidad_inicial.reindex(datos.columns)
calidad_final["ausentes_finales"] = datos.isna().sum()
calidad_final["porcentaje_ausente_final"] = 100 * calidad_final["ausentes_finales"] / len(datos)
calidad_final["caracter_danado_final"] = datos.astype("string").apply(
    lambda columna: columna.str.contains("\ufffd", regex=False, na=False).sum())
calidad_final["registros_con_cambio_contenido"] = cambios_contenido.sum()
calidad_final["tipo_final"] = datos.dtypes.astype(str)

resumen_procesamiento = pd.Series({
    "filas_originales": len(originales),
    "filas_base_comun": len(datos),
    "filas_eliminadas": len(originales) - len(datos),
    "filas_con_cambio_contenido": registros_modificados,
    "filas_sin_cambio_contenido": len(datos) - registros_modificados,
    "columnas_originales": originales.shape[1],
    "columnas_originales_conservadas": len(originales.columns.intersection(datos.columns)),
    "columnas_auxiliares": datos.shape[1] - originales.shape[1],
    "columnas_base_comun": datos.shape[1],
    "filas_con_asterisco_conservadas": int(datos["TIENE_ASTERISCO"].sum()),
    "repeticiones_adicionales_conservadas": int(repetida_adicional.sum()),
    "estadias_mayores_365_conservadas": int(datos["ESTADIA_MAYOR_365"].sum()),
    "filas_ERROR_1_conservadas": int(datos["ERROR"].eq(1).sum()),
    "DIAG1_sin_coincidencia_exacta": int(diagnosticos_sin_coincidencia["registros"].sum()),
    "DIAG2_sin_coincidencia_exacta_entre_presentes": int(sin_cruce_diag2["registros"].sum()),
    "celdas_numericas_no_convertibles": sum(valores_no_convertibles.values()),
}, name="cantidad").to_frame()
resumen_procesamiento.loc["comunas_con_caracteres_reparados", "cantidad"] = resumen_geografico.loc["CORREGIDA", "COMUNA"]
resumen_procesamiento.loc["regiones_con_caracteres_reparados", "cantidad"] = resumen_geografico.loc["CORREGIDA", "REGION"]
resumen_procesamiento.loc["comunas_con_nombre_normalizado", "cantidad"] = resumen_geografico.loc["NORMALIZADA", "COMUNA"]
resumen_procesamiento.loc["regiones_con_nombre_normalizado", "cantidad"] = resumen_geografico.loc["NORMALIZADA", "REGION"]
resumen_procesamiento.loc["filas_con_cambio_en_glosas_derivadas", "cantidad"] = int(cambios_geograficos.any(axis=1).sum())
resumen_procesamiento.loc["comunas_sin_correspondencia", "cantidad"] = resumen_geografico.loc["SIN_CORRESPONDENCIA", "COMUNA"]
resumen_procesamiento.loc["regiones_sin_correspondencia", "cantidad"] = resumen_geografico.loc["SIN_CORRESPONDENCIA", "REGION"]
resumen_procesamiento.loc["filas_con_glosas_corregidas_aun_danadas", "cantidad"] = int(datos["TEXTO_DANADO_PENDIENTE"].sum())
resumen_procesamiento["cantidad"] = resumen_procesamiento["cantidad"].astype("int64")

display(resumen_procesamiento)
display(calidad_final.round(3))

assert len(datos) == len(originales), "La preparación común debe conservar todos los egresos."
assert originales.columns.isin(datos.columns).all(), "Deben conservarse las 17 variables originales."
assert sum(valores_no_convertibles.values()) == 0, "Hay valores no convertibles que requieren revisión antes de exportar."

## 6. Exportación de una única base común

Parquet conserva tipos y ausentes. Se exporta una sola base con las variables originales, información
CIE-10 y marcas de calidad. El identificador es el número de fila del CSV, no un identificador de persona.
Las tablas documentan variables, calidad, transformaciones y verificaciones; los otros resultados
descriptivos permanecen visibles en el notebook. Las tres figuras se guardan para el informe.

In [ ]:
descripcion_auxiliares = {
    "ID_REGISTRO": "Número de fila del CSV, comenzando en 1; no identifica pacientes",
    "TIENE_ASTERISCO": "El registro tenía al menos un marcador * en la fuente",
    "TIENE_TEXTO_DANADO": "El registro tenía algún carácter de sustitución en la fuente",
    "EN_GRUPO_REPETIDO": "Los 17 campos originales coinciden con los de al menos otra fila",
    "DIAG2_VACIO_ORIGEN": "La causa externa estaba vacía en el CSV original",
    "TEXTO_DANADO_PENDIENTE": "Quedan caracteres de sustitución en las versiones corregidas de las glosas",
    "GLOSA_COMUNA_RESIDENCIA_CORREGIDA": "Nombre de comuna según CUT de la capa DPA 2023 de la SMA; conserva origen si no hay correspondencia",
    "GLOSA_REGION_RESIDENCIA_CORREGIDA": "Nombre de región según CUT de la capa DPA 2023 de la SMA; conserva origen si no hay correspondencia",
    "ESTADO_CORRECCION_COMUNA": "CORREGIDA, NORMALIZADA, COINCIDE, CATEGORIA_ESPECIAL, SIN_DATO o SIN_CORRESPONDENCIA",
    "ESTADO_CORRECCION_REGION": "CORREGIDA, NORMALIZADA, COINCIDE, CATEGORIA_ESPECIAL, SIN_DATO o SIN_CORRESPONDENCIA",
    "DIAG1_CATEGORIA": "Primeros tres caracteres del diagnóstico principal",
    "DIAG1_DESCRIPCION": "Descripción de subcategoría obtenida por coincidencia exacta",
    "DIAG1_GLOSA_CATEGORIA": "Descripción de categoría en el diccionario entregado",
    "DIAG1_CAPITULO": "Capítulo correspondiente a la categoría del diagnóstico principal",
    "DIAG1_GLOSA_CAPITULO": "Descripción del capítulo en el diccionario entregado",
    "DIAG1_COINCIDENCIA": "Coincidencia exacta, solo categoría, sin coincidencia o sin dato",
    "DIAG2_COINCIDENCIA": "Coincidencia exacta, sin coincidencia o sin dato en causa externa",
    "ESTADIA_MAYOR_365": "Más de 365 días; marca operativa de revisión, sin exclusión",
}
descripcion_final = descripcion_variables[["variable", "descripcion_diccionario"]].rename(
    columns={"descripcion_diccionario": "descripcion"})
auxiliares = pd.Series(descripcion_auxiliares, name="descripcion").rename_axis("variable").reset_index()
descripcion_final = pd.concat([descripcion_final, auxiliares], ignore_index=True).set_index("variable")
descripcion_final = descripcion_final.reindex(datos.columns)
descripcion_final["tipo_final"] = datos.dtypes.astype(str)
descripcion_final["origen"] = ["Fuente original" if columna in originales.columns else "Derivada" for columna in datos.columns]

datos.to_parquet(PROCESADOS / "egresos_2025_base_comun.parquet", index=False)
descripcion_final.to_csv(TABLAS / "descripcion_variables.csv", encoding="utf-8-sig")
calidad_final.to_csv(TABLAS / "calidad_datos.csv", encoding="utf-8-sig", float_format="%.6f")
transformaciones.to_csv(TABLAS / "transformaciones.csv", index=False, encoding="utf-8-sig")
diagnosticos_pendientes.to_csv(TABLAS / "diagnosticos_sin_coincidencia_exacta.csv", index=False, encoding="utf-8-sig")
resumen_procesamiento.to_csv(TABLAS / "resumen_procesamiento.csv", encoding="utf-8-sig")

print("Base común exportada:", (PROCESADOS / "egresos_2025_base_comun.parquet").relative_to(RAIZ))
print(f"Registros: {len(datos):,} | Atributos: {datos.shape[1]} | Tablas de documentación: 5 | Figuras: 3")

In [ ]:
display(datos.head(10))

El significado oficial de `*` y `ERROR` y los códigos diagnósticos sin correspondencia siguen pendientes.
Las glosas geográficas dañadas se conservan como origen y disponen de versiones corregidas por código. `ANO_EGRESO` queda ausente donde había `*`; no se imputa el año.
No se eliminaron repeticiones ni extremos y no se reconstruyeron datos clínicos faltantes.

Las preparaciones por técnica partirán de esta misma base. Las decisiones aprendidas de los datos
deberán ajustarse dentro de sus respectivos conjuntos de entrenamiento cuando corresponda.
El índice definitivo del avance depende de la pauta del profesor.

Fuente: ZIP y diccionario entregados por el equipo, identificados por SHA-256 al cargar.
[DEIS: definición de diagnóstico principal de egreso](https://repositoriodeis.minsal.cl/deis/rem_base/ver.asp?var3=9).